In [ ]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

In [ ]:
df["hubName"].value_counts()

In [ ]:
region_trends = (
    df.groupby(
        [
            "hubId",
            "hubName",
            "shopType",
            "skuNumber"
        ]
    )
    .agg(
        total_qty=(
            "effective_qty",
            "sum"
        ),

        unique_buyers=(
            "customerId",
            "nunique"
        ),

        total_orders=(
            "orderNumber",
            "nunique"
        )
    )
    .reset_index()
)

region_trends.head()

In [ ]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category"
        ]
    ]
    .drop_duplicates()
)

region_trends = (
    region_trends.merge(
        sku_lookup,
        on="skuNumber",
        how="left"
    )
)

region_trends.head()

In [ ]:
region_trends[
    "region_rank"
] = (
    region_trends
    .groupby(
        [
            "hubId",
            "shopType"
        ]
    )
    ["unique_buyers"]
    .rank(
        ascending=False,
        method="dense"
    )
)

region_trends.head()

In [ ]:
(
    region_trends[
        region_trends["hubName"]
        == "Instant Foods(Noida)"
    ]
    .sort_values(
        "region_rank"
    )
    [
        [
            "itemName",
            "unique_buyers",
            "region_rank"
        ]
    ]
    .head(10)
)

In [ ]:
region_trends.to_parquet(
    "../data/processed/region_trends.parquet",
    index=False
)

print("Saved")

In [ ]:
print(region_trends.shape)

print(
    region_trends["hubName"]
    .nunique()
)

print(
    region_trends["shopType"]
    .nunique()
)

In [ ]:
region_trends.shape

In [ ]:
region_trends.sort_values(
    "region_rank"
)[
    [
        "hubName",
        "shopType",
        "itemName",
        "region_rank"
    ]
].head(15)

In [ ]:
region_trends.shape